# Dhara — clean BGE-m3 retrieval fine-tuning

This notebook is for one claim only: improve **exact provision retrieval** without touching the frozen v2 test questions. It does not generate legal advice.

Run three experiments, selected only by `dev` Recall@10: base BGE-m3; LoRA on the approved pairs; and the same run with reminted hard negatives. Evaluate the selected checkpoint once on `test`.

> Do not upload the raw question files or this notebook's exported data to a public repository or a public Colab output.

## Required private input archive

Put a private `dhara_colab_v3.zip` in Google Drive. It must contain `data/processed/corpus_v1.jsonl`, `data/processed/train_retrieval_v5_negatives.jsonl`, `data/processed/dev_retrieval_v3.jsonl`, `data/processed/test_retrieval_v3.jsonl`, and optionally `data/processed/pretrain_retrieval_coverage_v1.jsonl`. Build the v5 file with `scripts/16_split_training.py` + `scripts/17_mine_negatives.py --variant strict` (see `merge_train_retrieval_v5.json` / `mine_negatives_v5.json`). Each retrieval row must have `qid`, `question`, `positive_chunk_ids`, and `hard_negative_chunk_ids` (ranks 5-30, globally excluded).

`train_retrieval_v5_negatives.jsonl` contains 2,685 approved pairs (`label_source == 'human_approved_title_pair'`, `source is None`), 972 LLM-authored pairs (`source == 'authored_v1'`), and 52 real human-adjudicated questions (`source == 'human_adjudicated_v2'`) — 3,709 rows total, each with 8 mined hard negatives. `test_retrieval_v3.jsonl` is the frozen 112-question human test. The optional coverage file is synthetic pretraining only: use it only after its audit passes and keep it as a separate ablation. Do **not** train on `train_retrieval_v4.jsonl` or `train_anchor_v1_negatives.jsonl`: both carry zero hard negatives and previously produced a null result.

In [1]:
# Runtime → Change runtime type → T4 GPU, then run this cell once.
# Colab currently ships torchao 0.10.0; recent PEFT detects it but requires >=0.16.
# torchao is optional for this LoRA run, so remove the incompatible optional package.
!nvidia-smi
!pip -q uninstall -y torchao
!pip -q install --upgrade --no-cache-dir 'sentence-transformers==3.4.1' 'transformers==4.48.3' 'peft==0.17.1' 'accelerate==1.4.0' datasets faiss-cpu

# Then use Runtime → Restart session. After restart, resume from the Drive-mount cell.

Wed Sep 16 13:37:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import zipfile

WORKDIR = Path('/content/drive/MyDrive/Dhara')
print(WORKDIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/Dhara


In [8]:
import json, random, re
from collections import Counter

DATA = WORKDIR
print(DATA)
def read_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

corpus = read_jsonl(DATA / 'corpus_v1.jsonl')
# v5 carries mined hard_negative_chunk_ids (ranks 5-30, globally excluded); v4 has none.
train = read_jsonl(DATA / 'train_retrieval_v5_negatives.jsonl')
dev = read_jsonl(DATA / 'dev_retrieval_v3.jsonl')
test = read_jsonl(DATA / 'test_retrieval_v3.jsonl')
approved_train = [r for r in train if r.get('label_source') == 'human_approved_title_pair' and r.get('source') is None]
human_train = [r for r in train if r.get('source') in ('human_adjudicated_v2', 'authored_v1')]
assert approved_train and human_train, 'Expected both approved and natural-human/authored supervision.'
assert all(r.get('hard_negative_chunk_ids') for r in train), 'v5 file must carry mined hard negatives for every row.'

def norm(s):
    return re.sub(r'\s+', ' ', s.casefold()).strip()

chunk = {r['chunk_id']: r for r in corpus}
for name, rows in [('train', train), ('dev', dev), ('test', test)]:
    assert rows, f'{name} is empty'
    for row in rows:
        assert row['question'].strip()
        assert row['positive_chunk_ids']
        assert set(row['positive_chunk_ids']) <= set(chunk), row['qid']

for left_name, left, right_name, right in [('train', train, 'dev', dev), ('train', train, 'test', test), ('dev', dev, 'test', test)]:
    assert not ({r['qid'] for r in left} & {r['qid'] for r in right}), f'qid leakage: {left_name}/{right_name}'
    assert not ({norm(r['question']) for r in left} & {norm(r['question']) for r in right}), f'question leakage: {left_name}/{right_name}'

covered_provisions = {chunk[c]['provision_id'] for r in train for c in r['positive_chunk_ids']}
print({'chunks': len(corpus), 'approved_title_train': len(approved_train), 'natural_human_train': len(human_train), 'dev': len(dev), 'test': len(test), 'approved_provisions': len(covered_provisions)})
assert len(approved_train) >= 2000 and len(human_train) >= 200 and len(covered_provisions) >= 1000, (
    'The approved final stage needs at least 2,000 queries covering 1,000+ provisions.'
)


/content/drive/MyDrive/Dhara
{'chunks': 39484, 'approved_train': 3000, 'dev': 246, 'test': 234, 'approved_provisions': 2356}


## Data gate

A 0.8 target is a **data-and-model** target. Aim for at least 3–5 independently phrased, reviewed questions per covered provision, broad Act coverage, and difficult same-Act / same-topic negatives. If the assertion above fails, stop training and grow the data; changing epochs will not make 168 covered provisions generalize across a 39k-chunk corpus.

In [ ]:
# Keep this template identical for training, indexing, and evaluation.
def document_text(row):
    title = row.get('act_title_bn') or row.get('act_title_en') or ''
    section = row.get('provision_no_ascii') or row.get('provision_no_bn') or ''
    return f"Act: {title} | Section: {section} | {row.get('text_raw', '')}"

def build_examples(rows):
    examples = []
    for r in rows:
        pos = document_text(chunk[r['positive_chunk_ids'][0]])
        # MNRL treats every batch positive as an in-batch negative. Mined hard
        # negatives (ranks 5-30, globally excluded) are added as explicit candidates.
        negatives = [document_text(chunk[c]) for c in r.get('hard_negative_chunk_ids', []) if c in chunk][:4]
        examples.append([r['question'], pos, *negatives])
    return examples

approved_examples = build_examples(approved_train)
human_examples = build_examples(human_train)
dev_examples = build_examples(dev)
len(approved_examples), len(human_examples), len(dev_examples)


In [ ]:
# Baseline evaluation. Record this before fine-tuning.
import numpy as np
from sentence_transformers import SentenceTransformer

BASE = 'BAAI/bge-m3'
EVAL_MAX_LENGTH = 384  # identical to LoRA training; never compare mismatched truncation.
def recall_metrics(model, rows, ks=(1, 5, 10, 100), batch_size=32):
    model.max_seq_length = EVAL_MAX_LENGTH
    max_k = max(ks)
    # Rank provisions, not paragraphs: one long section may have several chunks.
    candidate_depth = min(len(corpus), max_k * 20)
    docs = [document_text(r) for r in corpus]
    doc_ids = [r['chunk_id'] for r in corpus]
    provision_ids = [r['provision_id'] for r in corpus]
    provision_of_chunk = dict(zip(doc_ids, provision_ids))
    dv = model.encode(docs, batch_size=batch_size, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True)
    qv = model.encode([r['question'] for r in rows], batch_size=batch_size, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True)
    hits = {k: 0 for k in ks}
    precision = {k: 0.0 for k in ks}
    mrr10 = 0.0
    ndcg10 = 0.0
    for vector, row in zip(qv, rows):
        scores = vector @ dv.T
        top = np.argpartition(scores, -candidate_depth)[-candidate_depth:]
        ranked_chunks = top[np.argsort(scores[top])[::-1]]
        ranked = []
        seen = set()
        for idx in ranked_chunks:
            provision_id = provision_ids[idx]
            if provision_id not in seen:
                seen.add(provision_id)
                ranked.append(provision_id)
            if len(ranked) >= max_k:
                break
        gold = {provision_of_chunk[chunk_id] for chunk_id in row['positive_chunk_ids']}
        first_rank = next((rank for rank, provision_id in enumerate(ranked, 1) if provision_id in gold), None)
        for k in ks:
            found = set(ranked[:k]) & gold
            hits[k] += bool(found)
            # Diagnostic only: incomplete relevance labels make P@k a lower bound.
            precision[k] += len(found) / k
        if first_rank is not None and first_rank <= 10:
            mrr10 += 1 / first_rank
            ndcg10 += 1 / np.log2(first_rank + 1)
    metrics = {f'R@{k}': hits[k] / len(rows) for k in ks}
    metrics.update({f'P@{k}': precision[k] / len(rows) for k in ks})
    metrics['MRR@10'] = mrr10 / len(rows)
    metrics['nDCG@10'] = ndcg10 / len(rows)
    return metrics

base_model = SentenceTransformer(BASE, device='cuda')
base_model.max_seq_length = EVAL_MAX_LENGTH
base_dev = recall_metrics(base_model, dev)
base_test = recall_metrics(base_model, test)
baseline_dev = base_dev['R@10']
print('BASE dev :', base_dev)
print('BASE test:', base_test)

In [ ]:
# LoRA fine-tuning. T4-safe settings; increase epochs only if dev improves.
import importlib.util
assert importlib.util.find_spec('torchao') is None, 'torchao is still installed; rerun setup, then Runtime → Restart session.'
import torch
from datasets import Dataset
from peft import LoraConfig, TaskType, get_peft_model
from sentence_transformers import losses, SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.training_args import BatchSamplers

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
model = SentenceTransformer(BASE, device='cuda')
backbone = model[0].auto_model
lora = LoraConfig(task_type=TaskType.FEATURE_EXTRACTION, r=32, lora_alpha=64, lora_dropout=0.05, bias='none', target_modules=['query', 'key', 'value', 'dense'])
model[0].auto_model = get_peft_model(backbone, lora)
model[0].auto_model.print_trainable_parameters()

def as_dataset(examples):
    columns = {'anchor': [x[0] for x in examples], 'positive': [x[1] for x in examples]}
    if all(len(x) >= 3 for x in examples):
        columns['negative'] = [x[2] for x in examples]
    return Dataset.from_dict(columns)

def run_stage(name, dataset, epochs, learning_rate):
    args = SentenceTransformerTrainingArguments(
        output_dir=str(WORKDIR / 'outputs' / name), num_train_epochs=epochs,
        per_device_train_batch_size=16, gradient_accumulation_steps=2, batch_sampler=BatchSamplers.NO_DUPLICATES,
        learning_rate=learning_rate, warmup_ratio=0.1, fp16=True,
        save_strategy='epoch', logging_steps=20, report_to='none', seed=SEED,
    )
    trainer = SentenceTransformerTrainer(model=model, args=args, train_dataset=dataset, loss=losses.MultipleNegativesRankingLoss(model))
    trainer.train()

# Stage order matters: broad approved coverage first, then natural human questions
# closest to the held-out citizen-query distribution. Do not blend duplicate rows.
USE_APPROVED_COVERAGE_PRETRAIN = False  # Enable only after its audit passes; report it as an ablation.
if USE_APPROVED_COVERAGE_PRETRAIN:
    coverage = read_jsonl(DATA / 'pretrain_retrieval_coverage_v1.jsonl')
    run_stage('bge_m3_coverage_pretrain_v1', as_dataset(build_examples(coverage)), epochs=1, learning_rate=2e-6)
run_stage('bge_m3_approved_title_stage_v4', as_dataset(approved_examples), epochs=2, learning_rate=5e-6)
run_stage('bge_m3_natural_human_stage_v1', as_dataset(human_examples), epochs=3, learning_rate=3e-6)

In [ ]:
# Merge LoRA before saving. Saving the unmerged PEFT wrapper is invalid for a plain SentenceTransformer reload.
merged_backbone = model[0].auto_model.merge_and_unload()
model[0].auto_model = merged_backbone
OUT = WORKDIR / 'outputs' / 'bge_m3_retrieval_v2_merged'
model.save(str(OUT))
reloaded = SentenceTransformer(str(OUT), device='cuda')
reloaded.max_seq_length = EVAL_MAX_LENGTH
finetuned_dev_metrics = recall_metrics(reloaded, dev)
finetuned_dev = finetuned_dev_metrics['R@10']
print('BASE dev:', base_dev)
print('LoRA dev:', finetuned_dev_metrics)
assert finetuned_dev > baseline_dev, 'Do not promote this checkpoint. Mine better negatives or improve labels, then retrain.'

In [ ]:
# One-time paired frozen-test evaluation — base and LoRA use identical templates and truncation.
finetuned_test = recall_metrics(reloaded, test)
final_test_r10 = finetuned_test['R@10']
print('BASE test:', base_test)
print('LoRA test:', finetuned_test)
print(f'Paired R@10 delta: {final_test_r10 - base_test["R@10"]:+.4f}')
print('R@100 is the reranker ceiling: a reranker cannot recover a provision absent from top 100.')

# Save only the model and a small reproducibility record to private Drive.
import shutil
DEST = Path('/content/drive/MyDrive/dhara_private/experiments/bge_m3_retrieval_v2_merged')
if DEST.exists(): shutil.rmtree(DEST)
shutil.copytree(OUT, DEST)
(DEST / 'metrics.json').write_text(json.dumps({'base_dev': base_dev, 'base_test': base_test, 'finetuned_dev': finetuned_dev_metrics, 'finetuned_test': finetuned_test, 'seed': SEED}, indent=2), encoding='utf-8')
print('Saved privately to', DEST)

## If Recall@10 is still below target

Do not keep increasing epochs. Inspect 100 dev errors, then: (1) correct mislabeled positives, (2) add reviewed paraphrases for missed Acts/provisions, (3) re-mine 5–30 ranked hard negatives with the improved checkpoint while excluding every known relevant provision, and (4) train a BGE reranker on top-100 candidates using explicit relevant/not-relevant labels. A reranker cannot recover a gold provision absent from the first-stage top-100.

For a future hierarchical system, measure Act Recall@K first and union its candidates with global dense top-100. Never hard-filter to one predicted Act.